# Autoencoder Example — Compressing & Reconstructing Digits (MNIST)

A simple **autoencoder** built with TensorFlow / Keras. It trains in a few seconds on
CPU, which makes it a great companion to the heavier GAN example next door.

## What is an autoencoder?

An autoencoder is a neural network that learns to **copy its input to its output** —
which sounds pointless until you add a catch: it must squeeze the data through a narrow
**bottleneck** in the middle. Two halves:

- **Encoder** $f$: compresses a `784`-pixel image down to a tiny **latent code** (here,
  just 32 numbers).
- **Decoder** $g$: tries to rebuild the original image from that 32-number code.

Because all the information has to pass through the 32-dim bottleneck, the network is
forced to discover an efficient, lossy compression — it learns the *structure* of
handwritten digits (strokes, loops) rather than memorising pixels.

## How it's trained

Training is **unsupervised** — no labels needed. We simply minimise the **reconstruction
error** between the input image $x$ and its reconstruction $g(f(x))$:

$$\mathcal{L} = \| x - g(f(x)) \|^2 \quad\text{(mean squared error)}.$$

## How this differs from the GAN

Both models learn a low-dimensional code that maps to an image, but:

- A **GAN** learns to generate *new* images by fooling a discriminator — there is no
  direct target image, and it produces sharp but uncontrolled samples.
- An **autoencoder** has an explicit target (the input itself) and a clean training
  signal, so it is far easier and faster to train — but its outputs tend to be blurrier.

> Tested with `tensorflow==2.1` (matches `requirements.txt`). Trains in seconds on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)

## 1. Load and prepare the data

We scale pixels to `[0, 1]` (the decoder ends in a `sigmoid`, whose range is `[0, 1]`)
and **flatten** each `28x28` image into a `784`-vector, since this autoencoder is built
from plain dense layers. Again, **no labels** — an autoencoder only needs the images.

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print('Train:', x_train.shape, '| Test:', x_test.shape)

## 2. Build the autoencoder

We define three Keras models that share the same layers:

- **`encoder`**: `784 → 128 → 64 → 32`. Each ReLU dense layer shrinks the
  representation, funnelling the image down to the 32-dim **latent code**.
- **`decoder`**: `32 → 64 → 128 → 784`, mirroring the encoder, with a final `sigmoid`
  so every output pixel lands in `[0, 1]`.
- **`autoencoder`**: the encoder and decoder chained together — this is what we train.

Keeping a standalone `encoder` and `decoder` lets us inspect the codes and decode
arbitrary codes after training, not just run the full pipeline.

In [ ]:
LATENT_DIM = 32

# Encoder
encoder_input = tf.keras.Input(shape=(784,))
x = layers.Dense(128, activation='relu')(encoder_input)
x = layers.Dense(64, activation='relu')(x)
latent = layers.Dense(LATENT_DIM, activation='relu', name='latent')(x)
encoder = tf.keras.Model(encoder_input, latent, name='encoder')

# Decoder
decoder_input = tf.keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(64, activation='relu')(decoder_input)
x = layers.Dense(128, activation='relu')(x)
decoder_output = layers.Dense(784, activation='sigmoid')(x)
decoder = tf.keras.Model(decoder_input, decoder_output, name='decoder')

# Full autoencoder = decoder(encoder(x))
autoencoder = tf.keras.Model(encoder_input, decoder(encoder(encoder_input)), name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

## 3. Train

The key line is that the **input and the target are the same array** (`x_train,
x_train`) — that is what makes this 'self-supervised' reconstruction. We hold out the
test set as validation to confirm the model generalises rather than memorises. Unlike a
GAN, the loss here drops smoothly and monotonically.

In [ ]:
history = autoencoder.fit(
    x_train, x_train,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=2,
)

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.legend()
plt.title('Reconstruction loss'); plt.show()

## 4. Original vs reconstruction

The real test: feed test digits through the full autoencoder and compare each input
(top row) with its reconstruction from the 32-dim code (bottom row). They should be
clearly recognisable — slightly softened, since we threw away most of the 784 dimensions.

In [ ]:
n = 10
reconstructed = autoencoder.predict(x_test[:n])

plt.figure(figsize=(n * 1.5, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('original', loc='left')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('reconstructed', loc='left')
plt.show()

## 5. Bonus — generate digits from random codes

Since the decoder maps a 32-dim code to an image, we can hand it **random codes** and see
what it draws — a peek at the structure of the latent space. (The results are messier
than a GAN's: a plain autoencoder doesn't constrain the *shape* of its latent
distribution, so random points often miss the region the encoder actually uses. A
**variational** autoencoder fixes exactly this — a natural next step.)

In [ ]:
# Sample codes from the range the encoder actually produces on real data
codes = encoder.predict(x_test[:1000])
lo, hi = codes.min(0), codes.max(0)
random_codes = np.random.uniform(lo, hi, size=(10, LATENT_DIM)).astype('float32')
generated = decoder.predict(random_codes)

plt.figure(figsize=(15, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(generated[i].reshape(28, 28), cmap='gray'); plt.axis('off')
plt.suptitle('Digits decoded from random latent codes')
plt.show()

### Things to try

- Shrink `LATENT_DIM` to 8 or 2 and watch reconstructions get blurrier — a direct demo of
  the information bottleneck.
- Add noise to the input (`x_train + noise`) but keep the clean image as the target to
  build a **denoising autoencoder**.
- Swap the dense layers for `Conv2D` / `Conv2DTranspose` to make a convolutional
  autoencoder (sharper results on images).
- Turn it into a **variational autoencoder (VAE)** so the latent space becomes smooth and
  the random-code sampling above produces clean digits.